# Attention Mechanism

The decoder stops squinting at a compressed summary and starts looking at the whole souce

## Problem Definition

Every bit of information the encoder gleaned has to fit in one fixed-size hidden state.

Instead of giving the deocder only the final encoder state, keep every encoder state.

At each decoder step, compute a weighted average of encoder status, that weighted average is the context.

## Basic Concept

Decoder queries all encoder status, at each decoder step t:

1. Use the previous decoder hidden status s_{t-1} as a **query**.
2. Score it aganist every encoder hidden state h_1, ..., h_T. One scalar per encoder position.
3. Softmax the scores to get attention weights prob_{t, 1}, ..., prob_{t, T} that sum to 1.
4. Context vector is the weighted averge of encoder status.
5. Decoder takes weighted averge plus the previous output token, produces the next token.


**Bahdanau (additive) score.** `e_{t,i} = v_α^T * tanh(W_a * s_{t-1} + U_a * h_i)`.

- `s_{t-1}` has shape `(d_s,)`, `h_i` has shape `(d_h,)`.
- `W_a` has shape `(d_attn, d_s)`. `U_a` has shape `(d_attn, d_h)`.
- Their sum inside the tanh has shape `(d_attn,)`.
- `v_α` has shape `(d_attn,)`. The inner product with `v_α` collapses to a scalar. **This is what `v_α` does.** It is not magic. It is the projection that turns an attention-dim vector into a scalar score.

**Luong (multiplicative) score.** Three variants:

- `dot`: `e_{t,i} = s_t^T * h_i`. Requires `d_s == d_h`. Hard constraint. Skip if your encoder is bidirectional.
- `general`: `e_{t,i} = s_t^T * W * h_i` with `W` shape `(d_s, d_h)`. Removes the equal-dim constraint.
- `concat`: essentially the Bahdanau form. Rarely used since the first two are cheaper.

**One Bahdanau / Luong gotcha worth naming.** Bahdanau uses `s_{t-1}` (the decoder state *before* generating the current word). Luong uses `s_t` (the state *after*). Mixing them up produces subtly wrong gradients that are extremely hard to debug. Pick one paper and stick to its convention.

# Build your Own

## Bahdanau attention

In [1]:
import numpy as np

def softmax(x):
    x = x - np.max(x)
    e = np.exp(x)
    return e / e.sum()

def additive_attention(decoder_state, encoder_states, W_a, U_a, v_a):
    projected_dec = W_a @ decoder_state
    projected_enc = encoder_states @ U_a.T
    combined = np.tanh(projected_enc + projected_dec)
    scores = combined @ v_a
    weights = softmax(scores)
    context = weights @ encoder_states
    return context, weights

## Luong dot general

In [3]:
def dot_attention(decoder_state, encoder_states):
    scores = decoder_state @ encoder_states.T
    weights = softmax(scores)
    return weights @ encoder_states, weights

def general_attention(decoder_state, encoder_states, W):
    projected = W.T @ decoder_state
    scores = encoder_states @ projected
    weights = softmax(scores)

    return weights @ encoder_states, weights


H = np.array([
    [1.0, 0.0, 0.2],
    [0.5, 0.5, 0.1],
    [0.1, 0.9, 0.3],
])

s_close_to_cat = np.array([0.9, 0.1, 0.2])
ctx, w = dot_attention(s_close_to_cat, H)
print("weights:", w.round(3))

weights: [0.464 0.305 0.231]


## PyTorch Implement

In [ ]:
import torch
import torch.nn as nn

mha = nn.MultiheadAttention(embed_dim=128, num_heads=8, batch_first=True)
query = torch.randn(2, 5, 128)
key = torch.randn(2, 10, 128)
value = torch.randn(2, 10, 128)

output, weights = mha(query, key, value)
print(output.shape, weights.shape)